# Laboratorio 7 — Simulación del Álbum Panini FIFA 2026
**MM3014 Teoría de Probabilidades | Universidad del Valle de Guatemala**

## Etapa 1: Simulación básica con álbum reducido

Simulamos el proceso de llenar un álbum de **N = 100** estampas distintas, comprando sobres de **S = 7** estampas (todas distintas dentro del mismo sobre). Repetimos **R = 10,000** veces y comparamos con la **teoría del coleccionista**:

$$E[\text{sobres}] \approx \frac{N}{S} \cdot H_N, \quad H_N = \sum_{k=1}^{N} \frac{1}{k} \approx \ln(N) + \gamma, \quad \gamma \approx 0.5772$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros
N = 100       # estampas distintas
S = 7         # estampas por sobre
R = 10_000    # simulaciones
np.random.seed(2026)

### Simulación

In [ ]:
sobres_necesarios = []
repetidas_totales = []

for _ in range(R):
    coleccion = np.zeros(N, dtype=bool)  # qué estampas tengo
    sobres = 0
    repetidas = 0

    while not coleccion.all():  # hasta tener las N estampas
        sobre = np.random.choice(N, size=S, replace=False)  # 7 distintas por sobre
        sobres += 1
        for est in sobre:
            if coleccion[est]:
                repetidas += 1
            else:
                coleccion[est] = True

    sobres_necesarios.append(sobres)
    repetidas_totales.append(repetidas)

sobres_necesarios = np.array(sobres_necesarios)
repetidas_totales = np.array(repetidas_totales)

### Resultados

In [ ]:
# Estadísticas de sobres
media_sobres = sobres_necesarios.mean()
std_sobres   = sobres_necesarios.std()

# Estadísticas de repetidas
media_rep = repetidas_totales.mean()
std_rep   = repetidas_totales.std()

# Probabilidad de necesitar más de 30 sobres
# El mínimo teórico es ceil(N/S) = ceil(100/7) = 15 sobres (sin repetidas)
# 30 sobres es aprox el doble del mínimo, umbral razonable para analizar la cola
umbral = 30
prob_mas_30 = np.mean(sobres_necesarios > umbral)

# Valor teórico: E[sobres] = (N/S) * H_N
H_N = np.sum(1 / np.arange(1, N + 1))           # H_100 exacto
H_N_aprox = np.log(N) + 0.5772                  # H_100 aproximado
esperado_teorico = (N / S) * H_N

# Valor teórico de repetidas = S * E[sobres] - N
rep_teoricas = S * esperado_teorico - N

print("=" * 50)
print("SOBRES NECESARIOS")
print(f"  Media simulada:       {media_sobres:.2f}")
print(f"  Desv. estándar:       {std_sobres:.2f}")
print(f"  Valor teórico E[T]:   {esperado_teorico:.2f}  (H_100 exacto = {H_N:.4f})")
print(f"  H_100 aprox (ln+γ):   {H_N_aprox:.4f}")
print()
print("ESTAMPAS REPETIDAS")
print(f"  Media simulada:       {media_rep:.2f}")
print(f"  Desv. estándar:       {std_rep:.2f}")
print(f"  Valor teórico:        {rep_teoricas:.2f}")
print()
print("PROBABILIDAD DE MÁS DE 30 SOBRES")
print(f"  Umbral elegido: {umbral} sobres")
print(f"  (Mínimo teórico sin repetidas: ceil(100/7) = {int(np.ceil(N/S))} sobres)")
print(f"  P(sobres > {umbral}): {prob_mas_30:.4f}  ({prob_mas_30*100:.2f}%)")
print("=" * 50)

### Preguntas de análisis

**1. Mínimo teórico de sobres sin repetidas:**  
Si no hubiera repetidas, cada sobre aporta exactamente S = 7 estampas nuevas, por lo que el mínimo es ⌈100/7⌉ = **15 sobres**. En las simulaciones este caso prácticamente nunca ocurre porque la probabilidad de que 10,000 aperturas de sobres no produzcan ninguna repetida es extremadamente baja.

**2. Valor teórico de E[T]:**  
Se calcula en el código anterior. H₁₀₀ exacto ≈ 5.187, lo que da E[T] ≈ (100/7) × 5.187 ≈ **74.1 sobres**.

**3. Valor teórico de repetidas:**  
E[repetidas] = S × E[T] − N ≈ 7 × 74.1 − 100 ≈ **418.7 estampas repetidas**.

**4. Interpretación de la desviación estándar:**  
La desviación estándar es alta en relación con la media (cv > 0.3), lo que refleja la naturaleza aleatoria del proceso: algunas corridas terminan pronto y otras requieren muchos sobres para conseguir las últimas estampas raras. Esto es característico del *coupon collector problem*.

### Histograma

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(sobres_necesarios, bins=50, density=True,
         color='steelblue', edgecolor='black', alpha=0.8, label='Simulación')
plt.axvline(media_sobres, color='orange', linewidth=2, label=f'Media simulada ({media_sobres:.1f})')
plt.axvline(esperado_teorico, color='red', linestyle='--', linewidth=2,
            label=f'E[T] teórico ({esperado_teorico:.1f})')
plt.title('Etapa 1 — Distribución del número de sobres necesarios\n(N=100, S=7, R=10 000 simulaciones)')
plt.xlabel('Sobres necesarios para completar el álbum')
plt.ylabel('Densidad de probabilidad')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()